In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc

# -----------------------------
# Paths
# -----------------------------
BASE_ROOT = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs"

MODEL_DIRS = {
    "Genes_GBM": "Multirun_cell_on_cell_genes_new",
    "Genes_LASSO": "Multirun_cell_on_cell_genes_lasso",
    "Genes_ElasticNet": "Multirun_cell_on_cell_genes_elastic",
    "Genes_RandomForest": "Multirun_cell_on_cell_genes_rf",
    "Genes_Permuted": "Multirun_cell_on_cell_genes_permuted",
}

CELL_TYPES = ["Ast", "Ex", "In", "Oli", "Opc", "Mic"]
SPLITS = range(1, 6)

# -----------------------------
# Collect results
# -----------------------------
rows = []

for cell_type in CELL_TYPES:
    for model_name, model_dir in MODEL_DIRS.items():

        aucs = []

        base_path = os.path.join(BASE_ROOT, model_dir, cell_type)

        for i in SPLITS:
            pred_file = os.path.join(
                base_path, f"split_{i}", "test_predictions.csv"
            )

            if not os.path.exists(pred_file):
                continue

            df = pd.read_csv(pred_file)
            y_true = df["true_label"]
            y_score = df["predicted_proba"]

            fpr, tpr, _ = roc_curve(y_true, y_score)
            aucs.append(auc(fpr, tpr))

        if len(aucs) > 0:
            rows.append({
                "cell_type": cell_type,
                "model": model_name,
                "mean_auc": np.mean(aucs),
                "std_auc": np.std(aucs),
                "n_splits_used": len(aucs),
            })

# -----------------------------
# Final table
# -----------------------------
auc_table = pd.DataFrame(rows)

# Optional: sort nicely
auc_table = auc_table.sort_values(
    ["cell_type", "mean_auc"], ascending=[True, False]
)

# Save
out_path = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/"
    "genes_model_auc_summary.csv"
)
auc_table.to_csv(out_path, index=False)

print("Saved:", out_path)
print(auc_table)

Saved: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/genes_model_auc_summary.csv
   cell_type               model  mean_auc   std_auc  n_splits_used
1        Ast         Genes_LASSO  0.589058  0.054525              5
2        Ast    Genes_ElasticNet  0.575045  0.050046              5
0        Ast           Genes_GBM  0.567834  0.064186              5
3        Ast  Genes_RandomForest  0.561477  0.076015              5
4        Ast      Genes_Permuted  0.499498  0.146551              5
6         Ex         Genes_LASSO  0.606992  0.054479              5
7         Ex    Genes_ElasticNet  0.603144  0.053706              5
5         Ex           Genes_GBM  0.529827  0.042344              5
8         Ex  Genes_RandomForest  0.520718  0.075128              5
9         Ex      Genes_Permuted  0.468646  0.176697              5
13        In  Genes_RandomForest  0.655847  0.043110              5
10        In           Genes_GBM  0.627545  0.074425              5
11        In         Genes_LASS

In [2]:
# -----------------------------
# Summary counts
# -----------------------------
gbm_best_count = 0
permuted_worst_count = 0

for cell_type in CELL_TYPES:
    sub = auc_table[auc_table["cell_type"] == cell_type]

    # Best model (highest mean AUC)
    best_model = sub.loc[sub["mean_auc"].idxmax(), "model"]
    if best_model == "Genes_GBM":
        gbm_best_count += 1

    # Worst model (lowest mean AUC)
    worst_model = sub.loc[sub["mean_auc"].idxmin(), "model"]
    if worst_model == "Genes_Permuted":
        permuted_worst_count += 1

print(f"GBM best in {gbm_best_count} / {len(CELL_TYPES)} cell types")
print(f"Permuted worst in {permuted_worst_count} / {len(CELL_TYPES)} cell types")

GBM best in 2 / 6 cell types
Permuted worst in 6 / 6 cell types


In [3]:
print("\nBest and worst model per cell type:\n")

for cell_type in CELL_TYPES:
    sub = auc_table[auc_table["cell_type"] == cell_type]

    best_row = sub.loc[sub["mean_auc"].idxmax()]
    worst_row = sub.loc[sub["mean_auc"].idxmin()]

    print(
        f"{cell_type}: "
        f"BEST = {best_row['model']} (AUC={best_row['mean_auc']:.3f}), "
        f"WORST = {worst_row['model']} (AUC={worst_row['mean_auc']:.3f})"
    )


Best and worst model per cell type:

Ast: BEST = Genes_LASSO (AUC=0.589), WORST = Genes_Permuted (AUC=0.499)
Ex: BEST = Genes_LASSO (AUC=0.607), WORST = Genes_Permuted (AUC=0.469)
In: BEST = Genes_RandomForest (AUC=0.656), WORST = Genes_Permuted (AUC=0.461)
Oli: BEST = Genes_GBM (AUC=0.652), WORST = Genes_Permuted (AUC=0.486)
Opc: BEST = Genes_RandomForest (AUC=0.569), WORST = Genes_Permuted (AUC=0.481)
Mic: BEST = Genes_GBM (AUC=0.653), WORST = Genes_Permuted (AUC=0.489)


In [4]:
print("\nBest and worst model per cell type:\n")

for cell_type in CELL_TYPES:
    sub = auc_table[auc_table["cell_type"] == cell_type]

    best_row = sub.loc[sub["mean_auc"].idxmax()]
    worst_row = sub.loc[sub["mean_auc"].idxmin()]

    msg = (
        f"{cell_type}: "
        f"BEST = {best_row['model']} (AUC={best_row['mean_auc']:.3f}), "
        f"WORST = {worst_row['model']} (AUC={worst_row['mean_auc']:.3f})"
    )

    # If GBM is not best, report delta
    if best_row["model"] != "Genes_GBM":
        gbm_row = sub[sub["model"] == "Genes_GBM"]
        if not gbm_row.empty:
            delta = best_row["mean_auc"] - gbm_row.iloc[0]["mean_auc"]
            msg += f", Δ(best − GBM) = {delta:.3f}"

    print(msg)


Best and worst model per cell type:

Ast: BEST = Genes_LASSO (AUC=0.589), WORST = Genes_Permuted (AUC=0.499), Δ(best − GBM) = 0.021
Ex: BEST = Genes_LASSO (AUC=0.607), WORST = Genes_Permuted (AUC=0.469), Δ(best − GBM) = 0.077
In: BEST = Genes_RandomForest (AUC=0.656), WORST = Genes_Permuted (AUC=0.461), Δ(best − GBM) = 0.028
Oli: BEST = Genes_GBM (AUC=0.652), WORST = Genes_Permuted (AUC=0.486)
Opc: BEST = Genes_RandomForest (AUC=0.569), WORST = Genes_Permuted (AUC=0.481), Δ(best − GBM) = 0.028
Mic: BEST = Genes_GBM (AUC=0.653), WORST = Genes_Permuted (AUC=0.489)


In [5]:
import os

print("\n=== Checking split completeness per model and cell type ===\n")

missing_anything = False

for cell_type in CELL_TYPES:
    for model_name, model_dir in MODEL_DIRS.items():

        base_path = os.path.join(BASE_ROOT, model_dir, cell_type)
        missing_splits = []

        for i in SPLITS:
            pred_file = os.path.join(
                base_path, f"split_{i}", "test_predictions.csv"
            )
            if not os.path.exists(pred_file):
                missing_splits.append(i)

        if missing_splits:
            missing_anything = True
            print(
                f"[MISSING] Cell type = {cell_type}, Model = {model_name} "
                f"→ Missing splits: {missing_splits}"
            )

if not missing_anything:
    print("✅ All models have all 5 splits present for all cell types.")


=== Checking split completeness per model and cell type ===

✅ All models have all 5 splits present for all cell types.


In [2]:
import pandas as pd
import numpy as np

# -----------------------------
# Load AUC summary table
# -----------------------------
auc_table = pd.read_csv(
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/genes_model_auc_summary.csv"
)

# -----------------------------
# Compute ΔAUC = GBM − Permuted
# -----------------------------
rows = []

for cell_type in auc_table["cell_type"].unique():
    sub = auc_table[auc_table["cell_type"] == cell_type]

    gbm_auc = sub.loc[sub["model"] == "Genes_GBM", "mean_auc"].values
    perm_auc = sub.loc[sub["model"] == "Genes_Permuted", "mean_auc"].values

    if len(gbm_auc) == 1 and len(perm_auc) == 1:
        rows.append({
            "cell_type": cell_type,
            "gbm_auc": gbm_auc[0],
            "permuted_auc": perm_auc[0],
            "delta_auc_gbm_minus_permuted": gbm_auc[0] - perm_auc[0]
        })

delta_auc_df = pd.DataFrame(rows)

# -----------------------------
# Print per–cell-type ΔAUC
# -----------------------------
print("\nΔAUC (GBM − Permuted) by cell type")
print(delta_auc_df.sort_values("delta_auc_gbm_minus_permuted", ascending=False))

# -----------------------------
# Print summary statistics
# -----------------------------
mean_delta = delta_auc_df["delta_auc_gbm_minus_permuted"].mean()
std_delta = delta_auc_df["delta_auc_gbm_minus_permuted"].std()

print("\nSummary across cell types")
print(f"Mean ΔAUC (GBM − Permuted): {mean_delta:.3f} ± {std_delta:.3f}")


ΔAUC (GBM − Permuted) by cell type
  cell_type   gbm_auc  permuted_auc  delta_auc_gbm_minus_permuted
2        In  0.627545      0.461070                      0.166475
4       Oli  0.651956      0.485547                      0.166409
3       Mic  0.653109      0.488567                      0.164542
0       Ast  0.567834      0.499498                      0.068335
1        Ex  0.529827      0.468646                      0.061181
5       Opc  0.541218      0.481481                      0.059738

Summary across cell types
Mean ΔAUC (GBM − Permuted): 0.114 ± 0.056
